In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

## Load Cached Data from Local Storage

In [11]:
# META DATA CSV 

# check needed columns
metadata_cols = pd.read_csv('data/git_data/metadata.csv', nrows=0).columns
print(metadata_cols)

# load metadata as df
meta_df = pd.read_csv('data/git_data/metadata.csv',
                      usecols=['building_id', 'site_id', 'primaryspaceusage', 'sqm']
                     )

meta_df.head()

Index(['building_id', 'site_id', 'building_id_kaggle', 'site_id_kaggle',
       'primaryspaceusage', 'sub_primaryspaceusage', 'sqm', 'sqft', 'lat',
       'lng', 'timezone', 'electricity', 'hotwater', 'chilledwater', 'steam',
       'water', 'irrigation', 'solar', 'gas', 'industry', 'subindustry',
       'heatingtype', 'yearbuilt', 'date_opened', 'numberoffloors',
       'occupants', 'energystarscore', 'eui', 'site_eui', 'source_eui',
       'leed_level', 'rating'],
      dtype='str')


,building_id,site_id,primaryspaceusage,sqm
0,Panther_lodging_Dean,Panther,Lodging/residential,508.8
1,Panther_lodging_Shelia,Panther,Lodging/residential,929.0
2,Panther_lodging_Ricky,Panther,Lodging/residential,483.1
3,Panther_education_Rosalie,Panther,Education,690.5
4,Panther_education_Misty,Panther,Education,252.7


In [24]:
# filter building to include only Office or Education buildings

meta_df = meta_df[(meta_df['site_id'] == 'Panther') & 
    (meta_df['primaryspaceusage'].isin(['Education', 'Office']))]

print(meta_df.shape)
meta_df.head()

(63, 4)


,building_id,site_id,primaryspaceusage,sqm
3,Panther_education_Rosalie,Panther,Education,690.5
4,Panther_education_Misty,Panther,Education,252.7
5,Panther_office_Daina,Panther,Office,133.8
6,Panther_education_Mattie,Panther,Education,499.4
7,Panther_office_Woodrow,Panther,Office,537.8


In [30]:
# ELECTRICITY CSV

# check the electrical buildings csv for specific columns
elec_cols = pd.read_csv('data/git_data/electricity.csv', nrows=0).columns
print(len(elec_cols))

# search for number of columns with Panther and (Education or Office)
# include the timestamp column for analysis
search_pattern = r'^timestamp$|Panther.*(?:education|office)'
matched_cols = elec_cols[elec_cols.str.contains(search_pattern, regex=True)]

print(len(matched_cols))

1579
55


In [34]:
# load electric consumption profiles of education and office buildings
elec_df = pd.read_csv('data/git_data/electricity.csv',
                      index_col=0,
                      usecols=matched_cols,
                      parse_dates=True,
                     )

print(elec_df.shape)
elec_df.head()

(17544, 54)


,Panther_office_Hannah,Panther_education_Teofila,Panther_education_Jerome,Panther_education_Misty,Panther_office_Catherine,Panther_education_Tina,Panther_education_Janis,Panther_office_Patti,Panther_office_Lauretta,Panther_office_Valarie,...,Panther_education_Diann,Panther_education_Emily,Panther_education_Scarlett,Panther_education_Zelda,Panther_office_Jeane,Panther_office_Lavinia,Panther_office_Lois,Panther_education_Gina,Panther_education_Karri,Panther_education_Cleopatra
timestamp,,,,,,,,,,,,,,,,,,,,,
2016-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
# WEATHER CSV

# filter the columns of the weather csv

w_cols = pd.read_csv('data/git_data/weather.csv', nrows=0).columns
print(w_cols)

# site_id must correspond to Panther site
df_weather = pd.read_csv('data/git_data/weather.csv', 
                         index_col=0,
                         usecols=['timestamp', 'site_id', 'airTemperature', 'dewTemperature',],
                         parse_dates=True
                        )
df_weather = df_weather[df_weather['site_id']=='Panther']
print(df_weather.shape)
df_weather.head()

Index(['timestamp', 'site_id', 'airTemperature', 'cloudCoverage',
       'dewTemperature', 'precipDepth1HR', 'precipDepth6HR', 'seaLvlPressure',
       'windDirection', 'windSpeed'],
      dtype='str')
(17544, 3)


,site_id,airTemperature,dewTemperature
timestamp,,,
2016-01-01 00:00:00,Panther,19.4,19.4
2016-01-01 01:00:00,Panther,21.1,21.1
2016-01-01 02:00:00,Panther,21.1,21.1
2016-01-01 03:00:00,Panther,20.6,20.0
2016-01-01 04:00:00,Panther,21.1,20.6


### Education Buildings